In [34]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import GroupShuffleSplit

In [35]:
source = pd.read_json("python_dataset.jsonl", lines=True)

In [36]:
df = source

df = df.head(1000)

df = df.drop(columns="docstring")

# Convert df from wide format to long format. Originally it was idx, human_code, gpt_code, qwen_code ...
# But i need seperate rows per code snippet. With melt I transform it to idx, code, source
df = pd.melt(
    df,
    id_vars="hm_index",
    value_vars=["human_code", "chatgpt_code", "dsc_code", "qwen_code"],
    var_name="source",
    value_name="code",
)

df = df.dropna(axis=0)

# Add new column thats 0 or 1 depending on source. Human-written is 0, AI-generated is 1
df["ai_generated"] = df["source"].ne("human_code").astype(int)

df = df.rename(columns={"hm_index": "task_id"})

# Simplify task_id values and sort df
df["task_id"] = df.groupby("task_id").ngroup() + 1
df = df.sort_values(by=["task_id"])

# Move colum order so source comes last
df = df.iloc[:, [0, 2, 3, 1]]


In [ ]:
# Split code into input X and ouput y
X = df["code"]
y = df["ai_generated"]

# The df has multiple rows per coding task. e.g. task 1 has 4 rows (human-, gpt-, qwen- and dsc- code snippets)
# with GroupShuffleShift I split the the rows but dont put rows with the same task_id into train AND test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(df, groups=df["task_id"]):
    train_X, train_y = X.iloc[train_idx], y.iloc[train_idx]
    test_X, test_y = X.iloc[test_idx], y.iloc[test_idx]

# schneidung = train_X["task_id"].isin(test_X["task_id"]).any()
# print(schneidung)


transformer = TfidfTransformer(smooth_idf=False)

train_X = transformer.fit_transform(train_X)

##

ValueError: could not convert string to float: "def get_mnist(sc, data_type='train', location='/tmp/mnist'):\n    from pyspark.mllib.util import MLUtils\n    from pyspark.mllib.regression import LabeledPoint\n    from pyspark.sql import SparkSession\n    import os\n    spark = SparkSession(sc)\n    if not os.path.exists(location):\n        os.makedirs(location)\n    if not os.path.exists(os.path.join(location, f'{data_type}.data')):\n        import urllib.request\n        urllib.request.urlretrieve(f'http://yann.lecun.com/exdb/mnist/{data_type}-images-idx3-ubyte.gz', os.path.join(location, f'{data_type}-images-idx3-ubyte.gz'))\n        urllib.request.urlretrieve(f'http://yann.lecun.com/exdb/mnist/{data_type}-labels-idx1-ubyte.gz', os.path.join(location, f'{data_type}-labels-idx1-ubyte.gz'))\n    images_path = os.path.join(location, f'{data_type}-images-idx3-ubyte.gz')\n    labels_path = os.path.join(location, f'{data_type}-labels-idx1-ubyte.gz')\n    images_rdd = spark.sparkContext.bin